In [1]:
# Single-cell test for moltie/llm/client.py : verify_with_ollama()

from pathlib import Path
import sys, textwrap

CODE = Path("/home/hello/Projects/Statements/code").resolve()
PDF_PATH = Path("/home/hello/Projects/Statements/code/appeals/SAINSBURYS SUPERMARKETS LIMITED_vs_HITT.pdf").resolve()
assert CODE.exists(), CODE
assert PDF_PATH.exists(), PDF_PATH
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

# --- import only what we need ---
from moltie.llm.client import verify_with_ollama, LLMClientConfig
from moltie.schemas.verdict import Verdict
from moltie.agent.retrieve import retrieve_windowed_evidence
from moltie.corpus.chunker import iter_paragraphs
from moltie.schemas.query_object import AtomQuery

# --- PDF -> text fixture ---
try:
    from pypdf import PdfReader
except Exception:
    from PyPDF2 import PdfReader

text = "\n\n".join([(p.extract_text() or "") for p in PdfReader(str(PDF_PATH)).pages]).strip()
assert len(text) > 500, "PDF text extraction looks empty/too short (likely scanned OCR)."

doc_id = PDF_PATH.stem

# --- build paras [{para_id,text}] ---
paras = [{"para_id": f"p{i:05d}", "text": ptxt} for i, (_, _, ptxt) in enumerate(iter_paragraphs(text), start=1)]
assert len(paras) > 0

# --- build AtomQuery without guessing constructor fields ---
ann = getattr(AtomQuery, "__annotations__", {})
kwargs = {}
for k in ann.keys():
    if k == "atom_id": kwargs[k] = "X_TEST"
    elif k == "x_tests": kwargs[k] = ["X1"]
    elif k == "proposition": kwargs[k] = "Substitution / reasonableness test in unfair dismissal"
    elif k == "positive_indicators": kwargs[k] = ["substitute", "reasonable", "investigation", "unfair dismissal", "tribunal"]
    elif k in ("excludes", "keyword_seeds", "expansion_terms"): kwargs[k] = []
    else: kwargs[k] = None
atom = AtomQuery(**kwargs)

# --- retrieve evidence pack (proven working) ---
rr = retrieve_windowed_evidence(doc_id=doc_id, paras=paras, atom=atom, k=8, min_hits=2, window_size=24, stride=12, top_windows=2)
assert len(rr.paras) > 0

# --- build a minimal strict prompt (no verifier_prompt.py needed for this test) ---
# IMPORTANT: embed atom_id/doc_id as JSON-like keys so _extract_from_prompt can recover them.
evidence_lines = []
for p in rr.paras:
    evidence_lines.append(f'{p["para_id"]}: {p["text"]}')
evidence_blob = "\n\n".join(evidence_lines)

prompt = textwrap.dedent(f"""
You are a legal reasoning engine.
Return ONE JSON object ONLY matching the Verdict schema.

"atom_id": "{atom.atom_id}"
"doc_id": "{doc_id}"

Proposition:
{atom.proposition}

Positive indicators:
- {chr(10).join(atom.positive_indicators or [])}

Evidence paragraphs (ONLY use these; anchors must quote verbatim from them):
{evidence_blob}

Rules:
- If you cannot find verbatim anchors in the evidence, set relevant=false and anchors=[]
- If relevant=true, anchors must be non-empty
""").strip()

# --- call client ---
cfg = LLMClientConfig(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=450,
    max_retries=1,   # keep test tight
)

v = verify_with_ollama(prompt, cfg)

# --- assertions ---
assert isinstance(v, Verdict), type(v)
assert v.atom_id == atom.atom_id, (v.atom_id, atom.atom_id)
assert v.doc_id == doc_id, (v.doc_id, doc_id)

# schema-level invariants already enforced, but let's print proof
print("OK: got Verdict")
print("relevant:", v.relevant, "use_mode:", v.use_mode, "score/conf:", v.precedent_score, v.confidence)
print("anchors:", len(v.anchors))
if v.anchors:
    print("anchor[0]:", v.anchors[0].para_id)
    print("quote head:", v.anchors[0].quote[:140])
    # extra sanity: anchor para_id is in our evidence pack
    ev_ids = {p["para_id"] for p in rr.paras}
    assert v.anchors[0].para_id in ev_ids, f"Anchor para_id not in evidence pack: {v.anchors[0].para_id}"


RuntimeError: verify_with_ollama failed after 2 attempts. Last error: Expecting ',' delimiter: line 1 column 1516 (char 1515). Last output (truncated): '{\n  "atom_id": "X_TEST",\n  "doc_id": "SAINSBURYS SUPERMARKETS LIMITED_vs_HITT",\n  "precedent_score": 95,\n  "confidence": 90,\n  "relevant": true,\n  "anchors": [\n    "This case illustrates the dangers of encouraging an approach to unfair dismissal cases which leads an employment tribunal to substitute itself for the employer or to act as if it were conducting a rehearing of, or an appeal against, the merits of the employer\'s decision to dismiss. The employer, not the tribunal, is the proper person to conduct the investigation into the alleged misconduct. The function of the tribunal is to decide whether that investigation is reasonable in the circumstances and whether the decision to dismiss, in the light of the results of that investigation, is a reasonable response.",\n    "In the circumstances did the bank act reasonably or un